# exp/r2-ext on Colab (T4): queue [gru_v8p2z6a, gru_v8p2z6b]
Runtime -> Change runtime type -> T4 GPU, then Run all. A failed run raises with the config name and stops the queue (no silent skip). After a disconnect, just Run all again: each config resumes from its `last.pt` on Drive (`--resume`; model, optimizer, EMA, RNG), and a config whose `train_log.csv` already reached the final epoch or has an `early_stop` row is skipped. Each run writes `best.pt` (best **selection** PESQ, inference weights only; the EMA weights for z6b), `last.pt` (full state, every epoch) and `train_log.csv` (val / gen_unseen_dev / gen_multi_dev / selection / train64 PESQ per epoch); console output is appended to `train_stdout.log` there.

**Drive inputs** (both under `MyDrive/`): `noise_reduction_challenge/` (as before) and **`nrc_ext_z6/`** = the folder `data_ext/pkg/nrc_ext_z6` built by `scripts/gen/prepare_librispeech.py` + `scripts/gen/build_ext_package.py` (LibriSpeech subset, external noise train_pool, gen_unseen_dev + gen_multi_dev, MANIFEST.json). The setup cell copies it to `/content/nrc_ext_z6` and checks every SHA-256 in MANIFEST.json (fail loud).

Queue (2026-11-26, fifth batch), both continuing the submission gru_v8p2z3e (val 3.6534 / val_remix 3.4961; G_unseen_test 2.835, G_multi_test 2.240, G_low 2.311). Fresh Adam lr 5e-5, 0.2 cmag + 0.32 torch-pesq, train-time rms_norm 0.079130, z3e's speed (p 0.8, U[0.80, 1.20]) + EQ (p 0.8, +-12 dB) clean augmentation, 300 epochs x 81 steps, patience 40.
- **ext_mix** (both): clean = LibriSpeech train-clean-100 subset (WADA-SNR >= 30 dB, <= 40 utterances per speaker, random 3 s crop) p 0.5, else challenge train clean; noise = DEMAND/ESC-50/MUSAN **train_pool** (corpus uniform; never gen_dev/gen_test) p 0.6, else challenge train noise; SNR U[-10, 20] dB; second noise p 0.3 at U[-10, 0] dB relative.
- **Selection**: unweighted mean PESQ of validation (164), gen_unseen_dev (328) and gen_multi_dev (164), each logged separately.
- **gru_v8p2z6a** (seed 7): no EMA. **gru_v8p2z6b** (seed 8): EMA of weights, decay 0.999; validation and best.pt use the EMA weights.

Estimated T4 time: z3e's Colab log shows ~23.5 s training + 20.5 s evaluation (val + train64 = 228 files) per epoch. z6 evaluates 164 + 328 + 164 + 64 = 720 files (~65 s) and trains ~25 s, so **~90 s/epoch: up to ~7.5 h per config, ~15 h for the queue** (2-3 free sessions; Run all resumes), less if early stopping triggers. Setup (Drive copy + SHA-256 check of 3.4 GB) takes a few minutes per session.

After training: copy `nrc_ckpt/<name>/best.pt` to `weights/<name>.pt` (the configs point there) and `train_log.csv` to `logs/train_log_v8p2z6a.csv` / `logs/train_log_v8p2z6b.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
import os
!rm -rf /content/repo && git clone -b exp/r2-ext --depth 1 https://github.com/pauljh4323/SGSYS_13thTeam_nrc.git /content/repo
%cd /content/repo
w = 'weights/gru_v8p2z3e.pt'  # init_from of gru_v8p2z6a / gru_v8p2z6b
assert os.path.isfile(w) and os.path.getsize(w) > 1_000_000, f'{w} missing (init_from)'
print('init weights OK:', w, os.path.getsize(w), 'bytes')
print('CPUs:', os.cpu_count())

In [ ]:
!pip install -q pesq==0.0.4 pystoi soundfile pyyaml
# torch-pesq without its torchaudio dependency (src/nrc/tpesq.py provides the pieces it uses); see requirements-dpesq.txt
!pip install -q --no-deps -r requirements-dpesq.txt
import subprocess, sys
r = subprocess.run([sys.executable, '-c', 'import sys; sys.path.insert(0, "src"); from nrc.tpesq import pesq_loss; import torch; '
                    'l = pesq_loss(1.0, 16000); x = torch.randn(2, 48000) * 0.1; print("torch-pesq OK, MOS(x, x) =", l.mos(x, x).tolist())'],
                   capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
assert r.returncode == 0, 'torch-pesq import / smoke failed'

In [ ]:
# challenge data (as before) + the ext_mix package; every package file is checked against MANIFEST.json
import hashlib, json, shutil
from pathlib import Path
!mkdir -p /content/data && cp -r --update=none /content/drive/MyDrive/noise_reduction_challenge /content/data/
SRC_PKG, EXT_PKG = Path('/content/drive/MyDrive/nrc_ext_z6'), Path('/content/nrc_ext_z6')
assert (SRC_PKG / 'MANIFEST.json').is_file(), f'{SRC_PKG}/MANIFEST.json missing: upload data_ext/pkg/nrc_ext_z6 to MyDrive/nrc_ext_z6'
!cp -r --update=none /content/drive/MyDrive/nrc_ext_z6 /content/

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()

man = json.loads((EXT_PKG / 'MANIFEST.json').read_text())
bad = [rel for rel, (h, n) in man['files'].items()
       if not (EXT_PKG / rel).is_file() or (EXT_PKG / rel).stat().st_size != n or sha256(EXT_PKG / rel) != h]
assert not bad, f'{len(bad)} package files missing or corrupt, e.g. {bad[:5]}; delete /content/nrc_ext_z6 and re-copy (or re-upload)'
print(f"package OK: {len(man['files'])} files, {man['total_bytes'] / 1e9:.2f} GB; {man['librispeech']}")
# selection sets: the repo reads them from data/gen/<set> (nrc.io.EXTRA_SPLITS)
for s in ('gen_unseen_dev', 'gen_multi_dev'):
    dst = Path('data/gen') / s
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(EXT_PKG / 'gen' / s, dst)
    print(s, len(list((dst / 'clean').glob('*.wav'))), 'files')

In [ ]:
import os, yaml, pandas as pd
from pathlib import Path
CONFIGS = ["configs/dl/gru_v8p2z6a.yaml", "configs/dl/gru_v8p2z6b.yaml"]  # run in this order
for cfg in CONFIGS:
    assert os.path.isfile(cfg), f"{cfg} missing (wrong branch?)"
DATA = '/content/data/noise_reduction_challenge'
CKPT_ROOT = Path('/content/drive/MyDrive/nrc_ckpt')
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
for cfg in CONFIGS:
    lp = CKPT_ROOT / Path(cfg).stem / 'last.pt'
    print(Path(cfg).stem, 'resume point:', lp, 'found' if lp.is_file() else 'not found (fresh start from init_from)')
import subprocess, sys

def run_logged(cmd, log, name):
    """Run cmd, stream its output here and append it to log; raise if it exits non-zero (no silent `| tee`)."""
    print('$ ' + ' '.join(map(str, cmd)), flush=True)
    with open(log, 'a', encoding='utf-8') as f:
        proc = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='', flush=True)
            f.write(line)
        rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'{name} failed with exit code {rc}; see {log}')

In [ ]:
def ckpt_dir(cfg): return CKPT_ROOT / Path(cfg).stem

def finished(cfg):
    log = ckpt_dir(cfg) / 'train_log.csv'
    epochs = yaml.safe_load(open(cfg))['train']['epochs']
    if not log.is_file() or os.path.getsize(log) == 0:
        return False
    df = pd.read_csv(log)
    if len(df) == 0:
        return False
    early = 'early_stop' in df and (pd.to_numeric(df.early_stop, errors='coerce') == 1).any()
    return bool(early) or df.epoch.max() >= epochs

for cfg in CONFIGS:
    d = ckpt_dir(cfg)
    if finished(cfg):
        print(f'skip {cfg}: final epoch reached or early-stopped')
        continue
    d.mkdir(parents=True, exist_ok=True)
    run_logged([sys.executable, 'scripts/train_dpesq.py', '--config', cfg, '--data-root', DATA, '--ckpt-dir', d,
                '--ext-package', EXT_PKG, '--resume', '--device', 'auto'], d / 'train_stdout.log', Path(cfg).stem)

In [ ]:
for cfg in CONFIGS:
    log = ckpt_dir(cfg) / 'train_log.csv'
    if not log.is_file():
        print(f'{Path(cfg).stem}: no train_log.csv'); continue
    df = pd.read_csv(log).dropna(subset=['sel_pesq'])
    if df.empty:
        print(f'{Path(cfg).stem}: no validation yet'); continue
    r = df.loc[df.sel_pesq.idxmax()]
    print(f'{Path(cfg).stem}: best epoch {int(r.epoch)}, selection PESQ {r.sel_pesq:.4f} (val {r.val_pesq:.4f}, '
          f'gen_unseen_dev {r.gu_dev_pesq:.4f}, gen_multi_dev {r.gm_dev_pesq:.4f}), train64 {r.train64_pesq:.4f} '
          f'(gap {r.gap:+.4f}), val SNR {r.val_snr:.2f} dB (last epoch {int(pd.read_csv(log).epoch.max())})')